# Movio — Indic Voice Streaming Server (Kaggle GPU)

This notebook clones the Movio repo, installs dependencies, loads the TTS model on the Kaggle GPU, and exposes the server + frontend via a Cloudflare tunnel (free, no account needed).

**Just click Run All** — it handles install, kernel restart, model loading, and tunnel setup automatically.

After the last cell runs, click the printed `https://...trycloudflare.com` link to open the app.

## 1. Install & Auto-Restart

Installs dependencies on the first run (staged to avoid pip conflicts), then auto-restarts the kernel. On the second Run All it skips straight through.

In [ ]:
import os, sys

INSTALL_MARKER = "/tmp/.movio_installed"

if not os.path.exists(INSTALL_MARKER):
    print("First run — installing dependencies (staged to avoid conflicts)...")
    os.system("rm -rf /kaggle/working/movio-2")
    os.system("git clone https://github.com/tripathiji1312/movio_new.git /kaggle/working/movio-2")

    # Stage 1: parler-tts from git (pulls its own transformers/protobuf)
    os.system("pip install -q -U git+https://github.com/huggingface/parler-tts.git")
    # Stage 2: pin transformers to a version compatible with parler-tts's isin_mps_friendly import
    os.system('pip install -q "transformers==4.46.1"')
    # Stage 3: pin protobuf — Kaggle ships 4.x which is missing runtime_version (needs 5.x)
    os.system('pip install -q "protobuf==5.29.5"')
    # Stage 4: remaining deps (no conflicts)
    os.system("pip install -q -r /kaggle/working/movio-2/requirements.txt")
    os.system("pip install -q nest_asyncio")

    open(INSTALL_MARKER, "w").close()
    print("Install complete — kernel will restart now. Click Run All again.")
    print("(Next time it will skip install and go straight to model loading.)")

    # os._exit kills the process immediately so no subsequent cells run.
    # Kaggle auto-restarts the kernel after a crash.
    os._exit(0)

print("Dependencies already installed, skipping.")

## 2. Environment Check

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 3. HuggingFace Login

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("HuggingFace login successful!")

## 4. Load Model

Loads Indic Parler-TTS onto the GPU and runs a warm-up generation.

In [ ]:
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import sys
sys.path.insert(0, "/kaggle/working/movio-2")

from backend.tts_engine import engine
engine.load()

## 5. Start Server + Cloudflare Tunnel

Starts the FastAPI server (API + frontend) and creates a free Cloudflare tunnel.

Click the printed URL to open the app — it auto-connects, no pasting needed.

In [ ]:
import nest_asyncio
import uvicorn
import threading
import subprocess
import re
import time

from backend.server import app

nest_asyncio.apply()

PORT = 8000

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

# Download and start cloudflared (free, no account/token needed)
if not os.path.exists("/tmp/cloudflared"):
    os.system("wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
    os.system("chmod +x /tmp/cloudflared")

cf_process = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

tunnel_url = None
for _ in range(30):
    line = cf_process.stderr.readline().decode("utf-8", errors="replace")
    match = re.search(r"(https://[\w-]+\.trycloudflare\.com)", line)
    if match:
        tunnel_url = match.group(1)
        break

if tunnel_url:
    print("=" * 60)
    print("SERVER IS LIVE!")
    print("=" * 60)
    print(f"\nOpen this link in your browser:\n")
    print(f"  {tunnel_url}")
    print(f"\nThe frontend auto-connects — no URL pasting needed.")
    print("=" * 60)
else:
    print("ERROR: Could not get Cloudflare tunnel URL.")
    print("Stderr so far:")
    print(cf_process.stderr.read().decode("utf-8", errors="replace")[:2000])

## 6. (Optional) Quick Test

Quick local synthesis test to verify the model works.

In [ ]:
import numpy as np
from IPython.display import Audio, display
from backend.normalizer import normalize_text
from backend.chunker import split_into_sentences
from backend.config import VOICE_DESCRIPTION

test_text = "Your OTP is {{OTP:483927}}. Please do not share this with anyone."
chunks = [normalize_text(c) for c in split_into_sentences(test_text)]
print(f"Chunks: {chunks}")

for chunk in chunks:
    audio_pieces = []
    for piece, t in engine.stream_generate(chunk, VOICE_DESCRIPTION):
        audio_pieces.append(piece)
    audio = np.concatenate(audio_pieces)
    display(Audio(audio, rate=engine.sample_rate, autoplay=False))
    print(f"Generated {len(audio)/engine.sample_rate:.2f}s of audio")